# Bottlenecks & Capacity Strain Visualization

### Member 10 – Visualization

This notebook visualizes the key findings generated by Member 9's
Bottlenecks & Capacity Strain analysis.

### Business Questions
- Are there unusual patient length-of-stay patterns?
- Which doctors have the highest admission workload?
- Which departments show higher admission demand over time?
- Which departments have the highest critical-case concentration?

### Visualization Tool
Plotly

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [2]:
admissions = pd.read_csv("../data/processed/admissions_clean.csv")
doctors = pd.read_csv("../data/processed/doctors_preprocessed.csv")
departments = pd.read_csv("../data/processed/departments_clean.csv")

print("Admissions:", admissions.shape)
print("Doctors:", doctors.shape)
print("Departments:", departments.shape)

Admissions: (5000, 8)
Doctors: (500, 5)
Departments: (20, 2)


In [3]:
print("Admissions columns:")
print(admissions.columns.tolist())

print("\nDoctors columns:")
print(doctors.columns.tolist())

print("\nDepartments columns:")
print(departments.columns.tolist())

Admissions columns:
['Admission_ID', 'Patient_ID', 'Doctor_ID', 'Department_ID', 'Admission_Date', 'Discharge_Date', 'Status', 'Length_of_Stay_Days']

Doctors columns:
['Doctor_ID', 'Doctor_Name', 'Department_ID', 'Specialization', 'Experience']

Departments columns:
['department_id', 'department_name']


In [4]:
# Convert date columns to datetime
admissions["Admission_Date"] = pd.to_datetime(admissions["Admission_Date"])
admissions["Discharge_Date"] = pd.to_datetime(admissions["Discharge_Date"])

# Check for missing values
print("Missing values in Admissions:")
print(admissions.isnull().sum())

print("\nMissing values in Doctors:")
print(doctors.isnull().sum())

print("\nMissing values in Departments:")
print(departments.isnull().sum())

Missing values in Admissions:
Admission_ID           0
Patient_ID             0
Doctor_ID              0
Department_ID          0
Admission_Date         0
Discharge_Date         0
Status                 0
Length_of_Stay_Days    0
dtype: int64

Missing values in Doctors:
Doctor_ID         0
Doctor_Name       0
Department_ID     0
Specialization    0
Experience        0
dtype: int64

Missing values in Departments:
department_id      0
department_name    0
dtype: int64


In [5]:
department_map = departments.rename(columns={
    "department_id": "Department_ID"
})

admissions_dept = admissions.merge(
    department_map[["Department_ID", "department_name"]],
    on="Department_ID",
    how="left"
)

print(admissions_dept.head())

  Admission_ID Patient_ID Doctor_ID Department_ID Admission_Date  \
0       A00001     P00001   DR00186          D002     2025-10-20   
1       A00002     P00002   DR00281          D019     2025-01-21   
2       A00003     P00003   DR00105          D020     2025-06-24   
3       A00004     P00004   DR00312          D004     2025-07-23   
4       A00005     P00005   DR00430          D018     2025-04-06   

  Discharge_Date           Status  Length_of_Stay_Days department_name  
0     2025-10-28         Critical                    8       Neurology  
1     2025-01-23        Recovered                    2   Endocrinology  
2     2025-07-03       Discharged                    9          Dental  
3     2025-07-25         Critical                    2      Pediatrics  
4     2025-04-07  Under Treatment                    1   Ophthalmology  


In [6]:
los_summary = admissions_dept["Length_of_Stay_Days"].describe()

print(los_summary)

count    5000.000000
mean        5.515400
std         2.863669
min         1.000000
25%         3.000000
50%         6.000000
75%         8.000000
max        10.000000
Name: Length_of_Stay_Days, dtype: float64


In [7]:
Q1 = admissions_dept["Length_of_Stay_Days"].quantile(0.25)
Q3 = admissions_dept["Length_of_Stay_Days"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Q1: 3.0
Q3: 8.0
IQR: 5.0
Lower Bound: -4.5
Upper Bound: 15.5


In [8]:
los_outliers = admissions_dept[
    (admissions_dept["Length_of_Stay_Days"] < lower_bound) |
    (admissions_dept["Length_of_Stay_Days"] > upper_bound)
]

print("Number of LOS outliers:", len(los_outliers))

Number of LOS outliers: 0


In [9]:
fig_los = px.box(
    admissions_dept,
    y="Length_of_Stay_Days",
    points="outliers",
    title="Length of Stay Distribution"
)

fig_los.update_layout(
    title_x=0.5,
    yaxis_title="Length of Stay (Days)",
    xaxis_title="",
    template="plotly_white",
    height=450
)

fig_los.show()

In [10]:
# Doctor-wise admission count
doctor_workload = (
    admissions_dept.groupby("Doctor_ID")["Admission_ID"]
    .count()
    .reset_index(name="Admission_Count")
)

print(doctor_workload.head())

  Doctor_ID  Admission_Count
0   DR00001               14
1   DR00002               10
2   DR00003                6
3   DR00004               12
4   DR00005               11


In [11]:
doctor_workload = doctor_workload.merge(
    doctors[["Doctor_ID", "Doctor_Name", "Specialization"]],
    on="Doctor_ID",
    how="left"
)

print(doctor_workload.head())

  Doctor_ID  Admission_Count Doctor_Name Specialization
0   DR00001               14    Doctor_1            ENT
1   DR00002               10    Doctor_2      Emergency
2   DR00003                6    Doctor_3     Gynecology
3   DR00004               12    Doctor_4       Oncology
4   DR00005               11    Doctor_5         Dental


In [12]:
top_doctors = doctor_workload.sort_values(
    "Admission_Count",
    ascending=False
).head(10)

print(top_doctors)

    Doctor_ID  Admission_Count Doctor_Name    Specialization
38    DR00039               21   Doctor_39        Psychiatry
181   DR00182               19  Doctor_182            Dental
468   DR00469               18  Doctor_469       Orthopedics
29    DR00030               18   Doctor_30   General Surgery
14    DR00015               18   Doctor_15  Gastroenterology
272   DR00273               18  Doctor_273       Orthopedics
55    DR00056               17   Doctor_56     Endocrinology
310   DR00311               17  Doctor_311  Gastroenterology
369   DR00370               17  Doctor_370               ENT
396   DR00397               17  Doctor_397        Pediatrics


In [13]:
fig_doctor = px.bar(
    top_doctors.sort_values("Admission_Count"),
    x="Admission_Count",
    y="Doctor_Name",
    orientation="h",
    title="Top 10 Doctors by Admission Workload",
    hover_data=["Specialization"]
)

fig_doctor.update_layout(
    title_x=0.5,
    xaxis_title="Number of Admissions",
    yaxis_title="Doctor",
    template="plotly_white",
    height=500
)

fig_doctor.show()

In [14]:
# Monthly admissions by department

admissions_dept["Month"] = admissions_dept["Admission_Date"].dt.to_period("M").astype(str)

dept_monthly = (
    admissions_dept
    .groupby(["Month", "department_name"])
    .size()
    .reset_index(name="Admission_Count")
)

print(dept_monthly.head())

     Month department_name  Admission_Count
0  2025-01      Cardiology               19
1  2025-01          Dental               24
2  2025-01     Dermatology               13
3  2025-01             ENT               22
4  2025-01       Emergency               39


In [15]:
fig_dept = px.line(
    dept_monthly,
    x="Month",
    y="Admission_Count",
    color="department_name",
    markers=True,
    title="Monthly Admission Volume by Department"
)

fig_dept.update_layout(
    title_x=0.5,
    xaxis_title="Month",
    yaxis_title="Number of Admissions",
    template="plotly_white",
    height=550,
    legend_title="Department"
)

fig_dept.show()

In [16]:
# Admission status distribution

status_counts = (
    admissions_dept["Status"]
    .value_counts()
    .reset_index()
)

status_counts.columns = ["Status", "Admission_Count"]

print(status_counts)

            Status  Admission_Count
0         Critical             1279
1  Under Treatment             1272
2        Recovered             1235
3       Discharged             1214


In [17]:
total_admissions = len(admissions_dept)

critical_admissions = (
    admissions_dept["Status"]
    .eq("Critical")
    .sum()
)

critical_percentage = (
    critical_admissions / total_admissions
) * 100

print("Total Admissions:", total_admissions)
print("Critical Admissions:", critical_admissions)
print("Critical Admission Percentage:", round(critical_percentage, 2), "%")

Total Admissions: 5000
Critical Admissions: 1279
Critical Admission Percentage: 25.58 %


In [18]:
fig_status = px.pie(
    status_counts,
    names="Status",
    values="Admission_Count",
    hole=0.55,
    title="Admission Status Distribution"
)

fig_status.update_traces(
    textinfo="percent+label"
)

fig_status.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=500
)

fig_status.show()

## Critical Cases by Department

This visualization identifies departments with the highest number of critical admissions, highlighting areas with higher absolute operational pressure.

In [27]:
# Critical Cases by Department
critical_df = admissions_dept[admissions_dept["Status"] == "Critical"]
critical_by_dept = critical_df.groupby("department_name")["Admission_ID"].count().sort_values(ascending=False)

# Critical % by Department
total_by_dept = admissions_dept.groupby("department_name")["Admission_ID"].count()
critical_pct_by_dept = (critical_by_dept / total_by_dept * 100).sort_values(ascending=False).round(2)

# Average LOS by Department
avg_los_by_dept = admissions_dept.groupby("department_name")["Length_of_Stay_Days"].mean().sort_values(ascending=False).round(2)

# Top 10 longest stay patients
top_los = admissions_dept[["Patient_ID", "department_name", "Length_of_Stay_Days"]].sort_values(
    "Length_of_Stay_Days", ascending=False
).head(10)

print("Variables ready ✅")

Variables ready ✅


In [28]:
fig4 = px.bar(
    critical_by_dept.sort_values(ascending=True),
    orientation="h",
    title="Critical Cases by Department",
)
fig4.update_layout(
    title_x=0.5,
    xaxis_title="Number of Critical Admissions",
    yaxis_title="Department",
    template="plotly_white",
    height=450,
    showlegend=False
)
fig4.show()

In [29]:
fig5 = px.bar(
    critical_pct_by_dept.sort_values(ascending=True),
    orientation="h",
    title="Critical Case % by Department",
)
fig5.update_traces(texttemplate="%{x}%", textposition="outside")
fig5.update_layout(
    title_x=0.5,
    xaxis_title="Critical Case %",
    yaxis_title="Department",
    template="plotly_white",
    height=450,
    showlegend=False
)
fig5.show()

In [30]:
fig6 = px.bar(
    avg_los_by_dept.sort_values(ascending=True),
    orientation="h",
    title="Average Length of Stay by Department",
)
fig6.update_layout(
    title_x=0.5,
    xaxis_title="Average LOS (Days)",
    yaxis_title="Department",
    template="plotly_white",
    height=450,
    showlegend=False
)
fig6.show()

In [31]:
fig_table = go.Figure(data=[go.Table(
    header=dict(values=["Patient ID", "Department", "Length of Stay (Days)"],
                fill_color="#2C3E50", font=dict(color="white", size=12), align="left"),
    cells=dict(values=[top_los["Patient_ID"], top_los["department_name"], top_los["Length_of_Stay_Days"]],
               fill_color="#F5F5F5", align="left")
)])
fig_table.update_layout(title="Top 10 Longest-Stay Patients", title_x=0.5, template="plotly_white", height=400)
fig_table.show()

In [32]:
total_admissions = admissions_dept["Admission_ID"].nunique()
avg_los = admissions_dept["Length_of_Stay_Days"].mean().round(2)
critical_admissions = critical_df.shape[0]
top_doctor_workload = top_doctors["Admission_Count"].max()
top_doctor_name = top_doctors.sort_values("Admission_Count", ascending=False).iloc[0]["Doctor_Name"]

print(f"Total Admissions: {total_admissions}")
print(f"Average LOS: {avg_los} days")
print(f"Critical Admissions: {critical_admissions}")
print(f"Highest Doctor Workload: {top_doctor_workload} ({top_doctor_name})")

Total Admissions: 5000
Average LOS: 5.52 days
Critical Admissions: 1279
Highest Doctor Workload: 21 (Doctor_39)


In [33]:
from plotly.subplots import make_subplots

fig_kpi = make_subplots(
    rows=1, cols=4,
    specs=[[{"type": "indicator"}, {"type": "indicator"},
            {"type": "indicator"}, {"type": "indicator"}]]
)

# KPI 1: Total Admissions
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=total_admissions,
    title={"text": "Total Admissions"},
    number={"font": {"size": 40, "color": "#2C3E50"}}
), row=1, col=1)

# KPI 2: Average Length of Stay
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=avg_los,
    title={"text": "Average LOS (Days)"},
    number={"font": {"size": 40, "color": "#1F77B4"}, "suffix": " days"}
), row=1, col=2)

# KPI 3: Critical Admissions
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=critical_admissions,
    title={"text": "Critical Admissions"},
    number={"font": {"size": 40, "color": "#D62728"}}
), row=1, col=3)

# KPI 4: Highest Doctor Workload
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=top_doctor_workload,
    title={"text": f"Highest Doctor Workload<br><sub>{top_doctor_name}</sub>"},
    number={"font": {"size": 40, "color": "#2CA02C"}}
), row=1, col=4)

fig_kpi.update_layout(
    template="plotly_white",
    height=220,
    margin=dict(t=60, b=20, l=20, r=20),
    title_text="Key Performance Indicators",
    title_x=0.5
)

fig_kpi.show()

In [34]:
import plotly.io as pio

# Convert each figure to an HTML snippet (no need for full page each time)
kpi_html   = pio.to_html(fig_kpi, include_plotlyjs='cdn', full_html=False)
los_html   = pio.to_html(fig_los, include_plotlyjs=False, full_html=False)
doc_html   = pio.to_html(fig_doctor, include_plotlyjs=False, full_html=False)
dept_html  = pio.to_html(fig_dept, include_plotlyjs=False, full_html=False)
status_html= pio.to_html(fig_status, include_plotlyjs=False, full_html=False)
c4_html    = pio.to_html(fig4, include_plotlyjs=False, full_html=False)
c5_html    = pio.to_html(fig5, include_plotlyjs=False, full_html=False)
c6_html    = pio.to_html(fig6, include_plotlyjs=False, full_html=False)
table_html = pio.to_html(fig_table, include_plotlyjs=False, full_html=False)

# Combine everything into one dashboard page
dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Bottlenecks & Capacity Strain Dashboard</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #F5F6FA;
            margin: 0;
            padding: 20px;
        }}
        h1 {{
            text-align: center;
            color: #2C3E50;
        }}
        .chart-box {{
            background-color: white;
            border-radius: 10px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
            margin: 20px auto;
            padding: 10px;
            max-width: 1100px;
        }}
    </style>
</head>
<body>
    <h1>Bottlenecks & Capacity Strain Dashboard</h1>
    <div class="chart-box">{kpi_html}</div>
    <div class="chart-box">{los_html}</div>
    <div class="chart-box">{doc_html}</div>
    <div class="chart-box">{dept_html}</div>
    <div class="chart-box">{status_html}</div>
    <div class="chart-box">{c4_html}</div>
    <div class="chart-box">{c5_html}</div>
    <div class="chart-box">{c6_html}</div>
    <div class="chart-box">{table_html}</div>
</body>
</html>
"""

import os
os.makedirs('../outputs', exist_ok=True)

with open('../outputs/bottleneck_capacity_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

print("Dashboard HTML saved successfully at ../outputs/bottleneck_capacity_dashboard.html ✅")

Dashboard HTML saved successfully at ../outputs/bottleneck_capacity_dashboard.html ✅
